# Acknowledgment

This notebook is adapted from:

Heng-Jui Chang, NTUEE  
ML2021 Spring HW01  
Source: https://github.com/ga642381/ML2021-Spring/blob/main/HW01/HW01.ipynb

Modifications:
- Adjusted input feature construction
- Modified model architecture
- Integrated with our dataset pipeline

In [1]:
!nvidia-smi

Thu May  9 23:01:50 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 516.54       Driver Version: 516.54       CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ... WDDM  | 00000000:01:00.0 Off |                  N/A |
| N/A   39C    P8    N/A /  N/A |      0MiB /  2048MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

# Import packages

In [2]:
# Numerical Operations
import math
import numpy as np

# Reading/Writing Data
import pandas as pd
import os
import csv

# For Progress Bar
from tqdm import tqdm

# Pytorch
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

#For K-fold validation
from sklearn.model_selection import KFold

# Configurations
`config` contains hyper-parameters for training and the path to save your model.

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = {
    'seed': 5201314,     
    'select_all': True,   # Whether to use all features.
    'valid_ratio': 0.3,   # validation_size = train_size * valid_ratio
    'n_epochs':1000,            
    'batch_size': 2, 
    'learning_rate': 1e-5,              
    'early_stop': 15,        
    'save_path': './model.ckpt' 
}

def same_seed(seed): 
    '''Fixes random number generator seeds for reproducibility.'''
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
# Set seed for reproducibility
same_seed(config['seed'])

# Dataset

In [4]:
class GXData(Dataset):
    def __init__(self, x, y=None):
        if y is None:
            self.y = y
        else:
            self.y = torch.FloatTensor(y)
        self.x = torch.FloatTensor(x)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x[idx]
        else:
            return self.x[idx], self.y[idx]

    def __len__(self):
        return len(self.x)

In [5]:
def train_test_split(data_set, test_ratio, seed):
    '''Split provided training data into training set and test set'''
    test_set_size = int(test_ratio * len(data_set)) 
    train_set_size = len(data_set) - test_set_size
    train_set, test_set = random_split(data_set, [train_set_size, test_set_size], generator=torch.Generator().manual_seed(seed))
    return torch.FloatTensor(np.array(train_set) ), torch.FloatTensor( np.array(test_set) )

# Feature Selection
Choose features you deem useful by modifying the function below.

In [6]:
def select_feat(train_data, test_data, select_all=True):
    '''Selects useful features to perform regression'''
    y_train, y_test = train_data[:,-5:], test_data[:,-5:]
    raw_x_train, raw_x_test = train_data[:,:-5], test_data

    if select_all:
        feat_idx = list(range(raw_x_train.shape[1]))
    else:
        feat_idx = [0,1,2,3,4] # TODO: Select suitable feature columns.
        
    return raw_x_train[:,feat_idx], raw_x_test[:,feat_idx], y_train, y_test

# Dataloader
Read data from files and set up training, validation, and testing sets. You do not need to modify this part.

In [7]:
all_data = pd.read_csv('E:\\Homework\\natural_resources\\handin\\input_data.csv',sep = ',',header = None).values
train_data, test_data = train_test_split(all_data, config['valid_ratio'], config['seed'])

print(f"""train_data size: {train_data.shape} 
test_data size: {test_data.shape}""")

x_train, x_test, y_train, y_test = select_feat(train_data, test_data, config['select_all'])

print(f'number of features: {x_train.shape[1]}')

test_dataset = GXData(x_test, y_test)

test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, pin_memory=True)

train_data size: torch.Size([98, 12]) 
test_data size: torch.Size([42, 12])
number of features: 7


# Neural Network Model

In [8]:
class My_Model(nn.Module):
    def __init__(self, input_dim):
        super(My_Model, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 5),
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.squeeze(1) # (B, 1) -> (B)
        return x

# Training 

In [66]:
def trainer(x_train, y_train, model, config, device):

    criterion = nn.MSELoss(reduction='mean') 
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'] * 25000, weight_decay=1e-5 )  #20000,1e-5
    schedule = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0 = 4, T_mult=2, eta_min = config['learning_rate']*15000)  #15000

    if not os.path.isdir('./models'):
        os.mkdir('./models') # Create directory of saving models.

    n_epochs, best_loss, step, early_stop_count = config['n_epochs'], math.inf, 0, 0

    for epoch in range(n_epochs):
        model.train() # to train mode.
        loss_record = []        
        
        kfold = KFold(n_splits = 10)
        for train_index in kfold.split(x_train):
            #print(train_index[0])
            #print(train_index[1])
            #train_dataset = train_dataset[train_index]
            x_trainT = torch.index_select(x_train, 0, torch.IntTensor(train_index[0]) ); y_trainT = torch.index_select(y_train, 0, torch.IntTensor(train_index[0]) )
            x_trainV = torch.index_select(x_train, 0, torch.IntTensor(train_index[1]) ); y_trainV = torch.index_select(y_train, 0, torch.IntTensor(train_index[1]) )
            #train_loader = DataLoader(train_dataset, batch_size=train_dataset.__len__(), shuffle=True, pin_memory=True)
            #print(torch.IntTensor(train_index[0]) )
            #print(x_trainT.shape, y_trainT.shape)
            #print(x_trainV.shape, y_trainV.shape)
            #print("-----")
            

            
            optimizer.zero_grad()               # Set gradient to zero.
            x, y = x_trainT.to(device), y_trainT.to(device)   # Move your data to device. 
            pred = model(x)             
            loss = criterion(pred, y)
            loss.backward()                     # Compute gradient(backpropagation).
            optimizer.step()                    # Update parameters.
            step += 1
            loss_record.append(loss.detach().item())

        mean_train_loss = sum(loss_record)/len(loss_record)

        model.eval() # Set model to evaluation mode.
        loss_record = []
        
        x, y = x_trainV.to(device), y_trainV.to(device)
        with torch.no_grad():
            pred = model(x)
            loss = criterion( y, pred ) 
            loss_record.append(loss.item())
            
        mean_valid_loss = sum(loss_record)/len(loss_record)
        print(f'Epoch [{epoch+1}/{n_epochs}]: Train loss: {mean_train_loss:.4f}, Valid loss: {mean_valid_loss:.4f}')

        if mean_valid_loss < best_loss:
            best_loss = mean_valid_loss
            torch.save(model.state_dict(), config['save_path']) # Save best model
            print('Saving model with loss {:.3f}...'.format(best_loss))
            early_stop_count = 0
        else: 
            early_stop_count += 1

        if early_stop_count >= config['early_stop']:
            print('\nModel is not improving, so we halt the training session.')
            return
            
        schedule.step()

In [67]:
model = My_Model(input_dim=x_train.shape[1]).to(device)
trainer(x_train, y_train,  model, config, device)

Epoch [1/1000]: Train loss: 686.0485, Valid loss: 631.6079
Saving model with loss 631.608...
Epoch [2/1000]: Train loss: 491.8095, Valid loss: 496.8728
Saving model with loss 496.873...
Epoch [3/1000]: Train loss: 376.7884, Valid loss: 412.1829
Saving model with loss 412.183...
Epoch [4/1000]: Train loss: 307.6066, Valid loss: 354.0918
Saving model with loss 354.092...
Epoch [5/1000]: Train loss: 249.9889, Valid loss: 282.0902
Saving model with loss 282.090...
Epoch [6/1000]: Train loss: 196.7968, Valid loss: 227.9206
Saving model with loss 227.921...
Epoch [7/1000]: Train loss: 159.4625, Valid loss: 189.9383
Saving model with loss 189.938...
Epoch [8/1000]: Train loss: 134.2529, Valid loss: 164.2954
Saving model with loss 164.295...
Epoch [9/1000]: Train loss: 117.6585, Valid loss: 147.0741
Saving model with loss 147.074...
Epoch [10/1000]: Train loss: 106.8466, Valid loss: 135.4234
Saving model with loss 135.423...
Epoch [11/1000]: Train loss: 99.6984, Valid loss: 127.2163
Saving mod

# Testing
The predictions of your model on testing set will be stored at `pred.csv`.

In [68]:
def test(test_loader, model, device):
    loss = nn.MSELoss()
    ems = 0
    length = 0
    model.eval() # Set model to evaluation mode.
    preds = []
    for x, y in tqdm(test_loader):
        x = x.to(device)  
        length += x.shape[0]
        with torch.no_grad():                   
            pred = model(x)
            ems += loss(pred, y.to(device))
    ems /= length
    print( "ems is ",ems )
    return ems

In [69]:
model = My_Model(input_dim=x_train.shape[1]).to(device)
model.load_state_dict(torch.load(config['save_path']))
ems = test(test_loader, model, device)

100%|██████████| 21/21 [00:00<00:00, 663.63it/s]

ems is  tensor(31.5116, device='cuda:0')


In [13]:
print(model)

My_Model(
  (layers): Sequential(
    (0): Linear(in_features=7, out_features=5, bias=True)
  )
)


In [14]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

layers.0.weight tensor([[-2.6430e+00, -5.0549e-01,  5.8648e+01,  3.4390e+02,  1.2036e+01,
          8.5361e-01,  1.6603e+00],
        [-6.2930e+00,  1.1146e+01,  1.2072e-01, -3.9116e+02, -5.2737e+00,
          3.7356e+00,  1.2431e+00],
        [ 4.1419e+01, -4.4502e-01,  1.8897e+01, -4.2104e+02, -7.4002e+00,
          1.8537e+01, -5.6289e+00],
        [ 4.6398e+00,  5.3520e+01, -4.0849e+01,  8.7467e+01, -5.0165e-01,
         -2.8197e+01,  3.4299e-01],
        [-1.0823e+01,  1.0644e+01, -4.6314e+00, -1.4504e+02, -3.9064e-01,
         -4.2338e+00,  2.7926e+00]], device='cuda:0')
layers.0.bias tensor([25.7916,  4.1733, 13.1724,  5.8800,  0.7268], device='cuda:0')
